# 04 確認と補助の解析
実データのシャドー運用（誤差を入れずに当てはめ、QC の段差と照合）、管理限界の幅、Parvin 式との照合（実データ条件と合成の段階的照合）、
Alb の 1 週間の図データ、成分除去の要約、対応のある 95% 区間、バイアス感度の誤差の向き別を計算します。
所要時間の目安：デモ 1 分。実データは数分。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "pbrtqc").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 200)
from pbrtqc import config as C, pipeline
print(C.describe())

> **いまのモード**は上のセルの `profile=` で確認してください（`demo` = 合成データ、`real` = 実データ）。モードを変えたら、VS Code の「Restart」でカーネルを再起動してから上から実行し直します。

In [ ]:
log = pipeline.run([s[0] for s in pipeline.steps() if s[1] == "checks"], log_name="run_log_checks.json")

## Parvin 式との段階的照合（A は 1 前後になるはず）

In [ ]:
pd.read_csv(C.TABLES / "parvin_stages.csv")[["shift_sd", "A_ratio", "A_ratio_lo", "A_ratio_hi", "B_ratio", "C_ratio", "share_censored_C"]].round(3)

## 3 成分と 2 成分の比（対応のある 95% 区間）

In [ ]:
ci = pd.read_csv(C.TABLES / "paired_ci.csv")
ci[(ci["run"] == "fusion") & ci["B"].isin(["aod+out", "even+out"])].round(3)

## シャドー運用：年ごとの警報と QC 段差との一致

In [ ]:
sc = pd.read_csv(C.TABLES / "shadow_concordance.csv")
display(sc[sc["year"].astype(str).isin(["2023", "2024", "2025"])].groupby("year")["episodes"].sum())
x = pd.read_csv(C.TABLES / "shadow_2x2.csv")
print("QC 段差", int(x["qc_shifts"].sum()), "件、うち警報と一致", int(x["both"].sum()), "件")